# Analysis 2 · Path vs. filename & extensions

From the SURT path (query string excluded), split each URL into a **directory
path** vs. a trailing **filename with extension**.

**Extension definition (refined):** a trailing `.<2–4 chars>` counts as a file
extension only if it contains **at least one letter**. All-numeric endings like
`.2345` are *not* extensions — they're Lotus Notes element refs
(`Body/0.3528`), phone numbers (`507.835.8972`), ids (`/bia/0.001`), etc. — so
they stay on the directory-path side.

Deduplicated to one unique URL per `(crawl_year, surtkey)`. `.jpg` etc. are
counted as files and excluded from the path side. Companion: `a1_subdomains.ipynb`.

In [ ]:
import sys, os
os.umask(0o000)   # everything this notebook writes is world read/write (shared server, open to all)
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))
import config
import duckdb, pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_rows', 200); pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 90)
YEAR_ORDER = ['2004','2008','2012','2016','2020','2024']

def _writable_dir(preferred='deliverables'):
    for cand in (preferred, os.path.expanduser('~/eot_deliverables')):
        try:
            os.makedirs(cand, exist_ok=True)
            try: os.chmod(cand, 0o777)
            except OSError: pass
            t = os.path.join(cand, '.wtest'); open(t,'w').close(); os.remove(t)
            return cand
        except OSError:
            continue
    raise RuntimeError('no writable output dir found')
OUTDIR = _writable_dir()
print('OUTDIR =', os.path.abspath(OUTDIR))

DBS = config.discover_domain_dbs('cdxj')
print(f"{len(DBS)}/15 domain DBs found:", ", ".join(DBS) or "(none — run on the server)")

def per_domain(sql):
    frames = []
    for dom, path in DBS.items():
        con = duckdb.connect(str(path), read_only=True)
        df = con.sql(sql).df(); con.close()
        df.insert(0, 'domain', dom); frames.append(df)
    out = pd.concat(frames, ignore_index=True)
    if 'crawl_year' in out.columns:
        out['crawl_year'] = out['crawl_year'].astype(str)
    return out

def year_cols(df):
    return [y for y in YEAR_ORDER if y in df.columns]

def save(name):
    '''Output path; remove any stale file first so writes never hit the overwrite perm issue.'''
    p = os.path.join(OUTDIR, name)
    try:
        if os.path.exists(p): os.remove(p)
    except OSError: pass
    return p

## 2a. Path-only vs. with-filename split per year

`extension` = trailing `.<2–4 alnum>` **containing a letter**; else the URL is
directory-path-only.

In [ ]:
PATHFILE_SQL = r'''
WITH dedup AS (
  SELECT crawl_year, surtkey, min(regexp_extract(surtkey, '\)([^?]*)', 1)) AS url_path
  FROM eot_captures WHERE url NOT LIKE 'dns:%' GROUP BY 1, 2
),
cand AS (
  SELECT crawl_year,
    regexp_extract(url_path, '\.([a-zA-Z0-9]{2,4})$', 1) AS ext_raw,
    regexp_matches(url_path, '/[^/]+\.[a-zA-Z0-9]{2,4}$') AS has_trailing
  FROM dedup
),
paths AS (
  SELECT crawl_year,
    CASE WHEN has_trailing AND regexp_matches(ext_raw, '[a-zA-Z]')
         THEN lower(ext_raw) ELSE NULL END AS extension   -- numeric-only -> NULL (path, not file)
  FROM cand
)
SELECT crawl_year,
  count(*)                                              AS unique_urls,
  sum(CASE WHEN extension IS NULL THEN 1 ELSE 0 END)    AS path_only,
  sum(CASE WHEN extension IS NOT NULL THEN 1 ELSE 0 END) AS with_filename,
  round(100.0*sum(CASE WHEN extension IS NULL THEN 1 ELSE 0 END)/count(*),1) AS pct_path_only
FROM paths GROUP BY 1 ORDER BY 1
'''
split = per_domain(PATHFILE_SQL)
split.to_csv(save('a2_path_vs_filename.csv'), index=False)
split

## 2b. Path-only share over time (directory paths vs. files)

In [ ]:
piv = split.pivot(index='domain', columns='crawl_year', values='pct_path_only')
piv = piv[year_cols(piv)].reindex([d for d in config.TARGET_DOMAINS if d in piv.index])
plt.figure(figsize=(9,7))
sns.heatmap(piv, annot=True, fmt='.0f', cmap='Blues', vmin=0, vmax=100,
            cbar_kws={'label':'% URLs that are path-only (no filename)'}, linewidths=.5)
plt.title('Directory-path-only share of URLs, by domain and year')
plt.xlabel('crawl year'); plt.ylabel(''); plt.tight_layout()
plt.savefig(save('a2_path_only_pct.png'), dpi=150, bbox_inches='tight'); plt.show()

## 2c. Extension table (files only — letter-bearing extensions)

`.jpg`, `.pdf`, `.cfm`, … counted here; numeric-only endings excluded.

In [ ]:
EXT_SQL = r'''
WITH dedup AS (
  SELECT crawl_year, surtkey, min(regexp_extract(surtkey, '\)([^?]*)', 1)) AS url_path
  FROM eot_captures WHERE url NOT LIKE 'dns:%' GROUP BY 1, 2
),
cand AS (
  SELECT crawl_year, lower(regexp_extract(url_path, '\.([a-zA-Z0-9]{2,4})$', 1)) AS ext
  FROM dedup
  WHERE regexp_matches(url_path, '/[^/]+\.[a-zA-Z0-9]{2,4}$')
)
SELECT ext AS extension, crawl_year, count(*) AS n
FROM cand WHERE regexp_matches(ext, '[a-zA-Z]')   -- must contain a letter
GROUP BY 1, 2
'''
ext = per_domain(EXT_SQL)
ext.to_csv(save('a2_extensions_long.csv'), index=False)
ext_piv = ext.pivot_table(index='extension', columns='crawl_year', values='n', aggfunc='sum').fillna(0).astype(int)
ext_piv = ext_piv[year_cols(ext_piv)]; ext_piv['total'] = ext_piv.sum(axis=1)
ext_piv.sort_values('total', ascending=False).to_csv(save('a2_extensions_pivot.csv'))
print('Top letter-bearing file extensions across all domains:')
ext_piv.sort_values('total', ascending=False).head(30)

## 2d. QA — what the numeric-extension rule excludes

Example URLs whose trailing `.<2–4 digits>` was previously mis-counted as a file
and is now treated as directory path. Writes `a2_numeric_extension_examples.csv`
for review, and reports how many URLs this reclassifies.

In [ ]:
NUMEXT_SQL = r'''
WITH dedup AS (
  SELECT crawl_year, surtkey, min(url) AS url,
         min(regexp_extract(surtkey, '\)([^?]*)', 1)) AS url_path
  FROM eot_captures WHERE url NOT LIKE 'dns:%' GROUP BY 1, 2
),
num AS (
  SELECT crawl_year, url, surtkey,
         regexp_extract(url_path, '\.([a-zA-Z0-9]{2,4})$', 1) AS ext_raw
  FROM dedup
  WHERE regexp_matches(url_path, '/[^/]+\.[a-zA-Z0-9]{2,4}$')
    AND regexp_matches(regexp_extract(url_path, '\.([a-zA-Z0-9]{2,4})$', 1), '^[0-9]+$')
)
SELECT crawl_year, ext_raw AS numeric_ending, url, surtkey FROM num
'''
numext = per_domain(NUMEXT_SQL)
numext.to_csv(save('a2_numeric_extension_examples.csv'), index=False)
print(f"{len(numext):,} URLs reclassified from 'file' to 'path' by the letter rule "
      f"({numext.domain.nunique()} domains)")
print("Most common numeric endings:")
display(numext['numeric_ending'].value_counts().head(10).to_frame('n'))
numext[['domain','crawl_year','numeric_ending','url']].head(15)